# DAI Mission — Proposal Template
**Data & AI in Economics | TU Dortmund**

This notebook is your team's mission proposal. Fill in every section before submission. Once approved, you will extend this same notebook into your final deliverable.

> **Team size:** 2–3 students  
> **Deliverable:** This Jupyter Notebook (proposal → final submission in one file)


## 1. Team

| Role | Name | Student ID |
|------|------|------------|
| Lead | Masoumeh Davoudi | |
| Member | Armin Maddah Asl | |
| Member *(optional)* | Seyed Mohammad Hossein Ahmadi | |


## 2. Mission Title & Research Question

**Title:** *Education, Labor-Market Characteristics, and Income Inequality: Evidence from the Adult Census Income Dataset*

**Research question:**  
*To what extent is having at least a bachelor’s degree associated with the probability of earning more than `$50K` per year in the Adult Census Income dataset, and how do demographic, educational, occupational, and work-related characteristics jointly explain, predict, and structure high-income status?*

**Why it matters:**  
*Income inequality is a central topic in labor economics and social policy. While education is often considered an important determinant of earnings, income outcomes are also shaped by occupation, workclass, working hours, age, marital status, gender, race, country of origin, and capital-related income. By combining causal inference, supervised learning, and clustering, this project studies the adjusted relationship between education and high-income status, evaluates how well multiple labor-market and demographic features predict income groups, and identifies meaningful socio-economic profiles in the data.*


## 3. Data

**Source(s):**  
* Adult / Census Income dataset, provided by the UCI Machine Learning Repository. 
- Provider/source: UCI Machine Learning Repository; extracted by Barry Becker from the 1994 U.S. Census database.
- URL / access method: https://uci-ics-mlr-prod.aws.uci.edu/dataset/2/adult
- License: CC BY 4.0. The dataset contains 48,842 instances and 14 features.*

**Unit of observation:** *One row represents one individual person from the census-style data, including demographic, employment, education, and income-related characteristics.*

**Key variables:**


| Variable | Type | Role (feature / target / instrument / ...) | Description |
|----------|------|---------------------------------------------|-------------|
| `age` | Integer | Feature  | Age of the individual. No missing values. |
| `workclass` | Categorical | Feature  | Type of employment, such as `Private`, `Self-emp-not-inc`, `Self-emp-inc`, `Federal-gov`, `Local-gov`, `State-gov`, `Without-pay`, or `Never-worked`. Has missing values. |
| `fnlwgt` | Integer | Survey weight / feature | Final sampling weight used in the census-style dataset. No missing values. |
| `education` | Categorical | Feature   | Highest education level, such as `Bachelors`, `HS-grad`, `Masters`, `Doctorate`, `Some-college`, etc. No missing values. |
| `education-num` | Integer | Feature   | Numeric representation of education level. No missing values. |
| `marital-status` | Categorical | Feature  | Marital status, such as `Married-civ-spouse`, `Divorced`, `Never-married`, `Separated`, `Widowed`, etc. No missing values. |
| `occupation` | Categorical | Feature  | Type of occupation, such as `Tech-support`, `Sales`, `Exec-managerial`, `Prof-specialty`, `Craft-repair`, etc. Has missing values. |
| `relationship` | Categorical | Feature  | Family relationship category, such as `Wife`, `Husband`, `Own-child`, `Not-in-family`, `Other-relative`, or `Unmarried`. No missing values. |
| `race` | Categorical | Feature  | Race category, such as `White`, `Black`, `Asian-Pac-Islander`, `Amer-Indian-Eskimo`, or `Other`. No missing values. |
| `sex` | Binary categorical | Feature   | Sex of the individual: `Female` or `Male`. No missing values. |
| `capital-gain` | Integer | Feature | Capital gains reported by the individual. No missing values. |
| `capital-loss` | Integer | Feature | Capital losses reported by the individual. No missing values. |
| `hours-per-week` | Integer | Feature | Number of working hours per week. No missing values. |
| `native-country` | Categorical | Feature  | Country of origin, such as `United-States`, `Mexico`, `India`, `Canada`, `Germany`, etc. Has missing values. |
| `income` | Binary categorical | Target | Income class: `>50K` or `<=50K`. No missing values. |


**Potential data quality issues:**  
*  The Adult / Census Income dataset contains missing values in some categorical variables, especially `workclass`, `occupation`, and `native-country`. In the raw data, missing values may appear as `?`.

- The dataset is based on the 1994 U.S. Census, so the results may not reflect current labor-market conditions. Income levels, occupations, education returns, and demographic patterns may have changed since then.

- There may also be selection bias because the dataset is a filtered subset of census records. For example, the original extraction only includes individuals satisfying conditions such as age above 16, positive income, positive final weight, and positive working hours. Therefore, the dataset does not represent the entire population.

- For causal inference, the data is observational. Variables such as education, occupation, marital status, and working hours are not randomly assigned. This means that relationships such as education → income may be confounded by unobserved factors like family background, ability, local labor-market conditions, or access to opportunities.

- Some variables may also contain measurement or reporting issues. For example, `capital-gain` and `capital-loss` are highly skewed and may contain many zero values, while income is only given as a binary category (`>50K` or `<=50K`) instead of exact income. This limits the precision of income analysis.


In [6]:
# Data loading & first inspection
# ── replace with your own code ──────────────────────────────────────────
import pandas as pd
import numpy as np

column_names = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
    "income",
]

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

df = pd.read_csv(
    url, header=None, names=column_names, na_values="?", skipinitialspace=True
)

print("Shape of dataset:", df.shape)

display(df.head())

print("\nDataset information:")
print(df.info())

print("\nSummary statistics:")
display(df.describe())


missing_table = pd.DataFrame(
    {"missing_count": df.isna().sum(), "missing_share": df.isna().mean()}
).sort_values("missing_share", ascending=False)

print("\nMissing values:")
display(missing_table)

Shape of dataset: (32561, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       30725 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education-num   32561 non-null  int64 
 5   marital-status  32561 non-null  object
 6   occupation      30718 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital-gain    32561 non-null  int64 
 11  capital-loss    32561 non-null  int64 
 12  hours-per-week  32561 non-null  int64 
 13  native-country  31978 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB
None

Summary statistics:


,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000



Missing values:


,missing_count,missing_share
occupation,1843,0.056601
workclass,1836,0.056386
native-country,583,0.017905
fnlwgt,0,0.000000
education,0,0.000000
education-num,0,0.000000
age,0,0.000000
marital-status,0,0.000000
relationship,0,0.000000
sex,0,0.000000


## 4. Planned Methods

Your mission **must** apply at least one technique from **each** of the three blocks below. Tick the ones you plan to use and briefly justify the choice.

### 4a. Causal Inference
- [x] Causal graph / DAG (DoWhy)
- [x] Backdoor adjustment
- [ ] Instrumental variable
- [ ] Propensity score stratification
- [ ] Other: ___

*Justification:*  
We will estimate the effect of having at least a bachelor’s degree on earning over $50K. A DAG will identify likely confounders such as age, sex, race, and native country. We will use backdoor adjustment with regression controls as the main causal strategy.

### 4b. Supervised Learning
- [ ] Linear / Ridge / Lasso regression
- [x] Logistic regression
- [ ] k-Nearest Neighbors
- [ ] Support Vector Machine
- [x] Decision Tree / Random Forest
- [ ] Neural network (regression or classification)
- [ ] Other: ___

*Justification:*

### 4c. Unsupervised Learning / Generative Models
- [x] K-Means clustering
- [ ] Hierarchical clustering
- [ ] Variational autoencoder
- [ ] GAN
- [x] Other: PCA for dimensionality reduction and cluster visualization

*Justification:*  
We will use K-Means clustering to group individuals into similar socio-economic and labor-market profiles based on features such as age, education, occupation, workclass, and hours worked. The income variable will be excluded during clustering and used afterward to compare income patterns across clusters. PCA will help visualize the clusters.



## 5. Evaluation Strategy

*How will you know if your mission succeeded? Describe:*

- **The metric(s) you will use for each model:**  
  - For supervised learning, we will use accuracy, precision, recall, F1-score, ROC-AUC, and the confusion matrix to evaluate prediction of whether income is above `$50K`.  
  - For unsupervised learning, we will use the elbow method and silhouette score to evaluate clustering quality.   
  - For the causal inference component, we will report the estimated effect size, confidence intervals, and statistical significance of the education variable.

- **How you will validate causal claims:**    
  - We will validate the causal analysis by using a DAG to identify plausible confounders and by applying backdoor adjustment with regression controls. We will check robustness by comparing estimates across different model specifications, for example with and without selected control variables. We will also discuss possible remaining unobserved confounding, since education is not randomly assigned.

- **Any baselines or benchmarks you will compare against:**  
  - For supervised learning, we will compare our models against a majority-class baseline and a simple logistic regression model. 
  - For clustering, we will compare different values of `k` and choose the final number of clusters based on both silhouette score and interpretability. 
  - For causal inference, we will compare adjusted estimates with unadjusted estimates to show how controlling for confounders changes the estimated relationship.


## 6. Work Plan

| Step | Owner | Description |
|------|-------|-------------|
| 1 | Armin & Masoumeh| Data collection & cleaning |
| 2 | Masoumeh  | EDA |
| 3 | Armin | Causal inference block |
| 4 | Seyed Mohammad Hossein | Supervised learning block |
| 5 | Masoumeh | Unsupervised / generative block |
| 6 | All team members  | Synthesis & write-up |


---
## 7. Results *(complete for final submission)*


### 7a. Causal Inference

In [ ]:
# Causal inference analysis

### 7b. Supervised Learning

In [ ]:
# Supervised learning analysis

### 7c. Unsupervised / Generative

In [ ]:
# Unsupervised / generative analysis

## 8. Discussion & Conclusion *(complete for final submission)*

*Synthesise findings across all three method blocks. What does each lens reveal that the others miss? What are the limitations of your analysis?*
